# Delta Live Tables - Custos e Modos de Execução

> 🎯 **Foco na Certificação:** Este tópico é frequentemente cobrado na prova DEA!
---

## 📊 Visão Geral dos Modos

O DLT tem **dois eixos** de configuração independentes:

### Eixo 1: Development vs Production (Comportamento do Cluster)

| Modo | Objetivo | Custo | Cluster Shutdown |
|------|----------|-------|------------------|
| **Development** | Desenvolvimento e testes | 💰💰 Maior | 2 horas (padrão) |
| **Production** | Pipelines em produção | 💰 Menor | 0 segundos (imediato) |

### Eixo 2: Triggered vs Continuous (Modo de Execução)

| Modo | Objetivo | Custo | Execução |
|------|----------|-------|----------|
| **Triggered** | Batch, atualizações periódicas | 💰 Menor | Uma vez e para |
| **Continuous** | Streaming real-time | 💰💰💰 Maior | Sempre ativo |

---

## 🔧 Development Mode

### Características

| Característica | Comportamento |
|----------------|---------------|
| **Cluster Shutdown Delay** | **2 horas** (padrão) |
| **Reutilização de Cluster** | ✅ Sim - evita spin up/down |
| **Retries Automáticos** | ❌ Desabilitado |
| **Falha** | Imediata (sem retry) |

### Por que o cluster fica ativo 2 horas?

```
┌─────────────────────────────────────────────────────────────────┐
│                    DEVELOPMENT MODE                              │
├─────────────────────────────────────────────────────────────────┤
│                                                                  │
│   Run 1 ──► Run 2 ──► Run 3 ──► ... ──► 2h idle ──► Shutdown   │
│         │         │         │                                    │
│         └─────────┴─────────┘                                    │
│              Mesmo cluster (rápido!)                             │
│                                                                  │
│   ⚡ Vantagem: Iteração rápida durante desenvolvimento          │
│   💰 Desvantagem: Custo se deixar ocioso                        │
└─────────────────────────────────────────────────────────────────┘
```

### Quando usar?
- ✅ Durante **desenvolvimento ativo**
- ✅ Para **debug** e testes
- ✅ Quando precisa de **feedback rápido**
- ❌ **NÃO** usar em pipelines agendados

---

## 🏭 Production Mode

### Características

| Característica | Comportamento |
|----------------|---------------|
| **Cluster Shutdown Delay** | **0 segundos** (padrão) |
| **Reutilização de Cluster** | ❌ Não - novo cluster cada run |
| **Retries Automáticos** | ✅ Habilitado |
| **Recuperação de Falhas** | Automática (memory leaks, credentials) |

### Comportamento do cluster

```
┌─────────────────────────────────────────────────────────────────┐
│                    PRODUCTION MODE                               │
├─────────────────────────────────────────────────────────────────┤
│                                                                  │
│   Run 1: Start ──► Execute ──► Shutdown (imediato)              │
│                                                                  │
│   Run 2: Start ──► Execute ──► Shutdown (imediato)              │
│                                                                  │
│   Run 3: Start ──► Execute ──► Shutdown (imediato)              │
│                                                                  │
│   💰 Vantagem: Não paga por tempo ocioso                        │
│   ⏱️ Desvantagem: Startup time em cada execução                 │
└─────────────────────────────────────────────────────────────────┘
```

### Quando usar?
- ✅ Pipelines **agendados**
- ✅ Ambiente de **produção**
- ✅ Quando precisa de **estabilidade**
- ✅ Quando quer **otimizar custos**

---

## 💰 Comparação de Custos

### Custo por Modo

| Aspecto | Development | Production |
|---------|-------------|------------|
| **Cluster idle time** | Até 2 horas | 0 segundos |
| **Spin-up overhead** | Apenas 1x (reutiliza) | Cada execução |
| **Custo total (muitas runs)** | 💰💰 Maior | 💰 Menor |
| **Custo total (runs isoladas)** | 💰💰 Maior (espera 2h) | 💰 Menor |

### Cenário: 5 execuções em 1 hora

```
DEVELOPMENT MODE:
────────────────────────────────────────────────────────────
│ Run1 │ Run2 │ Run3 │ Run4 │ Run5 │   Idle (2h)    │ Shutdown
────────────────────────────────────────────────────────────
Custo: 3 horas de cluster (1h ativo + 2h idle)

PRODUCTION MODE:
────────────────────────────────────────────────────────────
│ Run1 │ │ Run2 │ │ Run3 │ │ Run4 │ │ Run5 │
────────────────────────────────────────────────────────────
     ↑       ↑       ↑       ↑       ↑
  Shutdown Shutdown Shutdown Shutdown Shutdown (imediato)

Custo: Apenas tempo de execução (~1h total)
```

---

## 📋 Configuração: pipelines.clusterShutdown.delay

Você pode customizar o tempo de shutdown do cluster:

```json
{
  "configuration": {
    "pipelines.clusterShutdown.delay": "30 minutes"
  }
}
```

| Modo | Valor Padrão |
|------|--------------|
| Development | `2 hours` |
| Production | `0 seconds` |

> 💡 **Dica:** Em Dev mode, você pode reduzir para `30 minutes` se quiser economizar.

---

## 💵 Pricing por Edição (DLT SKUs)

O custo também varia pela **edição** do DLT:

### Azure Databricks

| Edição | Preço/DBU | Recursos |
|--------|-----------|----------|
| **DLT Core** | $0.30 | Batch/streaming básico |
| **DLT Pro** | $0.38 | Core + CDC |
| **DLT Advanced** | $0.54 | Pro + Expectations + Quality Monitoring |

### AWS

| Edição | Preço/DBU | Recursos |
|--------|-----------|----------|
| **DLT Core** | $0.20 | Batch/streaming básico |
| **DLT Pro** | $0.25 | Core + CDC |
| **DLT Advanced** | $0.36 | Pro + Expectations + Quality Monitoring |

> ⚠️ **Importante para prova:** Se usar **Expectations** ou **CDC**, você precisa da edição **Pro** ou **Advanced**, que custam mais!

---

## 🔄 Triggered vs Continuous Mode

Além de Development/Production, existe outro eixo de configuração importante: **como** o pipeline executa.

### Comparação Detalhada

| Aspecto | Triggered | Continuous |
|---------|-----------|------------|
| **Execução** | Sob demanda ou agendada | Sempre ativo |
| **Cluster** | Inicia → Processa → Termina | Sempre rodando |
| **Latência** | Maior (startup time) | Menor (já está ativo) |
| **Custo** | 💰 Menor | 💰💰💰 Maior |
| **Caso de uso** | Batch, atualizações periódicas | Streaming real-time |
| **Novos dados** | Processados na próxima execução | Processados imediatamente |

### Triggered Mode (Padrão)

```
┌─────────────────────────────────────────────────────────────────┐
│                      TRIGGERED MODE                              │
├─────────────────────────────────────────────────────────────────┤
│                                                                  │
│   Schedule: Every 1 hour                                        │
│                                                                  │
│   00:00        01:00        02:00        03:00                  │
│     │            │            │            │                     │
│     ▼            ▼            ▼            ▼                     │
│   ┌───┐        ┌───┐        ┌───┐        ┌───┐                  │
│   │Run│        │Run│        │Run│        │Run│                  │
│   └───┘        └───┘        └───┘        └───┘                  │
│     │            │            │            │                     │
│   Stop         Stop         Stop         Stop                    │
│                                                                  │
│   ✅ Custo otimizado - paga apenas durante execução             │
└─────────────────────────────────────────────────────────────────┘
```

**Características:**
- Pipeline executa **uma vez** e depois para
- Pode ser agendado via **Databricks Jobs**
- Processa todos os dados disponíveis no momento da execução
- Cluster termina após conclusão (respeitando Dev/Prod mode)

**Quando usar:**
- ✅ Atualizações periódicas (diárias, horárias)
- ✅ Processamento batch
- ✅ Quando latência de minutos/horas é aceitável
- ✅ Otimização de custos

### Continuous Mode

```
┌─────────────────────────────────────────────────────────────────┐
│                     CONTINUOUS MODE                              │
├─────────────────────────────────────────────────────────────────┤
│                                                                  │
│   ══════════════════════════════════════════════════════════    │
│   │            Pipeline SEMPRE ATIVO                        │    │
│   ══════════════════════════════════════════════════════════    │
│                                                                  │
│   Dados chegam → Processados imediatamente → Disponíveis        │
│                                                                  │
│   Latência: segundos a minutos                                  │
│                                                                  │
│   ⚠️ Custo constante - cluster sempre consumindo DBUs          │
└─────────────────────────────────────────────────────────────────┘
```

**Características:**
- Pipeline fica **sempre ativo** aguardando novos dados
- Processa dados assim que chegam (near real-time)
- Usa micro-batches internamente (Structured Streaming)
- Cluster nunca termina (até você parar manualmente)

**Quando usar:**
- ✅ Streaming real-time
- ✅ Dashboards que precisam de dados atualizados
- ✅ Quando latência de segundos é necessária
- ✅ Pipelines de monitoramento/alertas

### Configuração

**Na UI:**
- Toggle "Continuous" nas configurações do pipeline

**Via JSON:**
```json
{
  "continuous": true   // ou false para Triggered
}
```

**Via API/CLI:**
```python
# Triggered (padrão)
{
  "continuous": false
}

# Continuous
{
  "continuous": true
}
```

### Impacto no Custo

```
┌─────────────────────────────────────────────────────────────────┐
│              COMPARAÇÃO DE CUSTO (24 horas)                     │
├─────────────────────────────────────────────────────────────────┤
│                                                                  │
│   TRIGGERED (execução a cada 1h, 10min cada):                   │
│   ────────────────────────────────────────────                  │
│   24 execuções × 10 min = 4 horas de compute                    │
│   Custo: ~4 horas de DBU                                        │
│                                                                  │
│   CONTINUOUS:                                                    │
│   ────────────────────────────────────────────                  │
│   24 horas × cluster ativo = 24 horas de compute                │
│   Custo: ~24 horas de DBU                                       │
│                                                                  │
│   📊 Continuous pode custar 6x mais neste exemplo!              │
└─────────────────────────────────────────────────────────────────┘
```

### Combinação com Dev/Prod Mode

| Triggered + Dev | Triggered + Prod | Continuous + Dev | Continuous + Prod |
|-----------------|------------------|------------------|-------------------|
| Executa → Cluster fica 2h → Para | Executa → Para imediato | Sempre ativo (ignora 2h) | Sempre ativo |
| 💰💰 | 💰 | 💰💰💰 | 💰💰💰 |

> ⚠️ **Importante:** Em Continuous mode, o `pipelines.clusterShutdown.delay` não se aplica pois o cluster nunca para!

---

## ✅ Checklist para Prova

### Development Mode (Classic)
- [ ] Cluster fica ativo por **2 horas** (padrão) ✅ *Confirmado no Lakeflow 2025*
- [ ] **Reutiliza** cluster entre execuções
- [ ] Retries **desabilitados** (falha imediata)
- [ ] Usado para **desenvolvimento e testes**
- [ ] Custo **maior** se deixar ocioso

### Production Mode (Classic)
- [ ] Cluster termina **imediatamente** (0 segundos) ✅ *Confirmado no Lakeflow 2025*
- [ ] **Novo cluster** a cada execução
- [ ] Retries **habilitados** (recuperação automática)
- [ ] Usado para **pipelines agendados**
- [ ] Custo **menor** (não paga idle time)

### Triggered Mode
- [ ] Executa **uma vez** e para
- [ ] Pode ser **agendado** via Jobs
- [ ] Custo **menor** (paga só execução)
- [ ] **Padrão** quando não especificado

### Continuous Mode
- [ ] Pipeline **sempre ativo**
- [ ] Processa dados **imediatamente** (near real-time)
- [ ] Custo **maior** (cluster nunca para)
- [ ] `clusterShutdown.delay` **não se aplica**

### Serverless
- [ ] **Não tem** Dev/Prod mode (gerenciado automaticamente)
- [ ] Requer **Unity Catalog** (obrigatório)
- [ ] **Elastic billing** (paga só tempo de processamento)
- [ ] Dois modos: **Standard** e **Performance Optimized**
- [ ] Standard: startup **4-6 minutos**, custo **menor**
- [ ] Performance Optimized: startup **rápido**, custo **maior**
- [ ] Standard mode **só funciona com Triggered**
- [ ] Continuous **sempre usa Performance Optimized**

### Configuração
- [ ] `pipelines.clusterShutdown.delay` customiza o tempo ✅ *Só Classic*
- [ ] `"continuous": true/false` no JSON
- [ ] Toggle fácil na UI (não afeta código)

### Edições (SKUs)
- [ ] **Core** = básico (mais barato)
- [ ] **Pro** = Core + CDC
- [ ] **Advanced** = Pro + Expectations (mais caro)

---

## 🔥 Pegadinhas Comuns

| Pergunta | Resposta |
|----------|----------|
| Qual modo é mais barato para produção? | **Production** (Classic) ou **Serverless Standard** |
| Por que Dev mode tem cluster ativo 2h? | Para **reutilizar** e acelerar iteração |
| Production mode tem retry automático? | **SIM** |
| Development mode tem retry automático? | **NÃO** (falha imediata) |
| Mudar Dev/Prod afeta o código? | **NÃO** (só comportamento do cluster) |
| Expectations aumentam o custo? | **SIM** (requer Advanced edition) |
| Qual o shutdown delay padrão em Prod? | **0 segundos** |
| Qual modo de execução é o padrão? | **Triggered** |
| Continuous mode respeita shutdown delay? | **NÃO** (cluster nunca para) |
| Qual modo tem menor latência? | **Continuous** |
| Qual modo tem menor custo? | **Triggered** |
| Serverless tem Dev/Prod mode? | **NÃO** (automático) |
| Serverless requer Unity Catalog? | **SIM** (obrigatório) |
| Serverless Standard funciona com Continuous? | **NÃO** (só Triggered) |
| O que é elastic billing? | Paga **só tempo de processamento** |

---

## 🎯 Resumo Visual

```
┌─────────────────────────────────────────────────────────────────┐
│                    DOIS EIXOS DE CONFIGURAÇÃO                   │
├─────────────────────────────────────────────────────────────────┤
│                                                                  │
│   EIXO 1: Development vs Production (comportamento do cluster)  │
│   ─────────────────────────────────────────────────────────────  │
│   Development: Cluster reutilizado, 2h idle, sem retries        │
│   Production:  Cluster termina imediato, com retries            │
│                                                                  │
│   EIXO 2: Triggered vs Continuous (modo de execução)            │
│   ─────────────────────────────────────────────────────────────  │
│   Triggered:  Executa uma vez e para (batch)                    │
│   Continuous: Sempre ativo, processa em real-time               │
│                                                                  │
└─────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────┐
│                    MATRIZ DE CUSTO                              │
├─────────────────────────────────────────────────────────────────┤
│                                                                  │
│                    │  Triggered      │  Continuous              │
│   ─────────────────┼─────────────────┼─────────────────────────  │
│   Development      │  💰💰 Médio     │  💰💰💰 Alto             │
│                    │  (2h idle)      │  (sempre ativo)          │
│   ─────────────────┼─────────────────┼─────────────────────────  │
│   Production       │  💰 Baixo       │  💰💰💰 Alto             │
│                    │  (termina logo) │  (sempre ativo)          │
│                                                                  │
│   🏆 Menor custo: Production + Triggered                        │
│   ⚡ Menor latência: Continuous (qualquer)                      │
└─────────────────────────────────────────────────────────────────┘
```

---

## 🔗 Referências

- [Run an update on a DLT pipeline](https://docs.databricks.com/en/delta-live-tables/updates.html)
- [Configure pipeline settings](https://learn.microsoft.com/en-us/azure/databricks/delta-live-tables/settings)
- [Configure classic compute for pipelines](https://docs.databricks.com/aws/en/ldp/configure-compute) *(Lakeflow - mais recente)*
- [DLT Pricing](https://www.databricks.com/product/pricing/delta-live)

---

## 🆕 Novidades Lakeflow (2025)

Com a evolução para **Lakeflow Spark Declarative Pipelines**, algumas otimizações de custo foram adicionadas:

| Novidade | Impacto no Custo |
|----------|------------------|
| **Predictive Optimization** | OPTIMIZE e VACUUM automáticos baseados em padrões de uso (menos compute manual) |
| **Serverless Pipelines** | Opção serverless disponível (não requer permissão de compute) |
| **Enhanced Autoscaling** | Melhor escalonamento automático |
| **Vertical Autoscaling** | Detecta out-of-memory e ajusta instance types automaticamente |

---

## ☁️ Serverless Pipelines

O Serverless **muda completamente** o paradigma de custos e configuração!

### Classic vs Serverless

| Aspecto | Classic Compute | Serverless |
|---------|-----------------|------------|
| **Dev/Prod Mode** | ✅ Configurável | ❌ N/A (automático) |
| **Cluster Shutdown Delay** | ✅ Configurável | ❌ N/A (elástico) |
| **Permissão necessária** | Compute creation | Nenhuma especial |
| **Configuração de cluster** | Manual (JSON) | Automático |
| **Unity Catalog** | Opcional | **Obrigatório** |
| **Billing** | Tempo de cluster | Tempo de processamento real |
| **Autoscaling** | Enhanced | Enhanced + Vertical |

### Serverless Performance Modes

O Serverless tem **seus próprios modos** de performance:

| Modo | Startup | Custo DBU | Caso de Uso |
|------|---------|-----------|-------------|
| **Standard** | 4-6 minutos | 💰 Menor | Latência tolerável |
| **Performance Optimized** | Rápido (segundos) | 💰💰 Maior | Time-sensitive |

> ⚠️ **Importante:** Standard mode **só funciona com Triggered** pipelines. Continuous sempre usa Performance Optimized!

### Benefícios do Serverless

```
┌─────────────────────────────────────────────────────────────────┐
│                    SERVERLESS BILLING                           │
├─────────────────────────────────────────────────────────────────┤
│                                                                  │
│   CLASSIC COMPUTE:                                              │
│   ════════════════════════════════════════════════════════      │
│   │ Startup │   Processing   │    Idle    │ Shutdown │          │
│   ════════════════════════════════════════════════════════      │
│         ▲              ▲            ▲                           │
│         │              │            │                           │
│     💰 Paga       💰 Paga      💰 Paga (Dev mode)              │
│                                                                  │
│   SERVERLESS:                                                   │
│   ════════════════════════════════════════════════════════      │
│   │        │   Processing   │          │          │             │
│   ════════════════════════════════════════════════════════      │
│                    ▲                                            │
│                    │                                            │
│               💰 Paga APENAS aqui (elastic billing)             │
│                                                                  │
│   📊 Resultado: Até 98% de economia em transformações complexas │
└─────────────────────────────────────────────────────────────────┘
```

### Features Exclusivas do Serverless

| Feature | Descrição |
|---------|-----------|
| **Stream Pipelining** | Micro-batches executam em paralelo (não sequencial) |
| **Incremental MV Refresh** | Materialized Views atualizam incrementalmente |
| **Vertical Autoscaling** | Ajusta instance types para evitar OOM |
| **Elastic Billing** | Paga apenas tempo de processamento real |

### Quando Usar Cada Um?

```
┌─────────────────────────────────────────────────────────────────┐
│                    DECISÃO: CLASSIC vs SERVERLESS               │
├─────────────────────────────────────────────────────────────────┤
│                                                                  │
│   Use SERVERLESS quando:                                        │
│   ✅ Usar Unity Catalog                                         │
│   ✅ Quer simplicidade (zero config)                            │
│   ✅ Workloads variáveis (elastic billing)                      │
│   ✅ Equipe sem permissão de compute                            │
│   ✅ Quer otimização automática                                 │
│                                                                  │
│   Use CLASSIC quando:                                           │
│   ✅ Precisa de Hive Metastore (legado)                         │
│   ✅ Requer configuração específica de cluster                  │
│   ✅ Private networking customizado                             │
│   ✅ Workloads muito previsíveis (commitment pricing)           │
│                                                                  │
└─────────────────────────────────────────────────────────────────┘
```

### Matriz de Custo Completa (2025)

| Compute | Execution Mode | Performance | Custo Relativo |
|---------|---------------|-------------|----------------|
| Classic + Prod | Triggered | N/A | 💰 Baixo |
| Classic + Dev | Triggered | N/A | 💰💰 Médio |
| Classic | Continuous | N/A | 💰💰💰 Alto |
| **Serverless** | Triggered | **Standard** | 💰 **Menor** |
| Serverless | Triggered | Performance | 💰💰 Médio |
| Serverless | Continuous | Performance | 💰💰💰 Alto |

> 🏆 **Menor custo:** Serverless + Triggered + Standard mode
> ⚡ **Melhor performance:** Serverless + Continuous + Performance Optimized